# 🛰️ Agricultural Field Parcel Segmentation
### Computer Vision Analysis of Satellite Imagery
---
**Author:** Zia U. Ahmed, Ph.D. | Uppata Analytics LLC  
**Methods:** Felzenszwalb · SLIC · Watershed · K-Means · Hough Lines  
**Target:** Agricultural parcel delineation from high-resolution satellite imagery

> This notebook demonstrates a multi-method computer vision pipeline for segmenting  
> agricultural field boundaries from satellite imagery using classical CV techniques.


## 1. Install & Import Dependencies

In [ ]:
# Install required packages (run once)
# !pip install opencv-python-headless scikit-image matplotlib numpy pillow scipy

import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import matplotlib.gridspec as gridspec

from skimage.segmentation import felzenszwalb, slic, mark_boundaries
from skimage.color import label2rgb
from skimage import img_as_float
from skimage.measure import regionprops, label as sk_label

from scipy import ndimage
from PIL import Image
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ All packages loaded successfully.")
print(f"   OpenCV     : {cv2.__version__}")
print(f"   NumPy      : {np.__version__}")
print(f"   Matplotlib : {matplotlib.__version__}")


## 2. Configuration & Colour Palette

In [ ]:
# ── IMAGE PATH ────────────────────────────────────────────────────────────────
IMG_PATH = "your_satellite_image.png"   # ← replace with your image path

# ── SEGMENTATION PARAMETERS ───────────────────────────────────────────────────
CONFIG = {
    "fz_scale_fine"   : 80,    # Felzenszwalb fine-scale
    "fz_scale_mid"    : 200,   # Felzenszwalb mid-scale  (primary)
    "fz_scale_coarse" : 400,   # Felzenszwalb coarse-scale
    "fz_sigma"        : 1.0,
    "fz_min_size_fine": 200,
    "fz_min_size_mid" : 400,
    "fz_min_size_crs" : 700,
    "slic_n_segments" : 120,
    "slic_compactness": 18,
    "slic_sigma"      : 0.8,
    "kmeans_k"        : 6,
    "hough_threshold" : 55,
    "hough_min_length": 30,
    "hough_max_gap"   : 15,
    "edge_canny_low"  : 15,
    "edge_canny_high" : 50,
    "ws_fg_threshold" : 0.35,  # watershed foreground distance threshold
    "min_parcel_pct"  : 1.5,   # % of image area to consider a major parcel
}

# ── COLOUR PALETTE (20 perceptually distinct colours) ─────────────────────────
PALETTE = [
    (255,210, 60), ( 60,200,120), (255,100, 80), ( 80,160,255),
    (210, 80,210), (255,175, 40), ( 50,215,200), (240, 80, 80),
    (130,205, 55), (160, 95,255), (255,235,100), ( 40,185,255),
    (200,135, 50), ( 95,240,130), (255,125,175), (170,195, 55),
    (100,180,240), (240,160, 50), ( 55,200,160), (200, 60,120),
]

BG_DARK  = '#080810'
BG_PANEL = '#0f0f1a'
TEXT_COL = '#dde2ff'
SUB_COL  = '#8899bb'

print("✅ Configuration set.")
print(f"   FZ scales    : {CONFIG['fz_scale_fine']} / {CONFIG['fz_scale_mid']} / {CONFIG['fz_scale_coarse']}")
print(f"   SLIC segments: {CONFIG['slic_n_segments']}")
print(f"   K-Means k    : {CONFIG['kmeans_k']}")


## 3. Load & Inspect Image

In [ ]:
img_bgr = cv2.imread(IMG_PATH)
if img_bgr is None:
    raise FileNotFoundError(f"Image not found: {IMG_PATH}")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W    = img_rgb.shape[:2]
img_f   = img_as_float(img_rgb)
gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
hsv     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

print(f"Image loaded: {W} × {H} px  ({W*H:,} total pixels)")
print(f"Channels    : {img_rgb.shape[2]}")
print(f"dtype       : {img_rgb.dtype}")

# Quick display
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG_DARK)
axes[0].imshow(img_rgb); axes[0].axis('off')
axes[0].set_title('RGB', color=TEXT_COL, fontsize=11)

axes[1].imshow(gray, cmap='gray'); axes[1].axis('off')
axes[1].set_title('Grayscale', color=TEXT_COL, fontsize=11)

plt.suptitle('Input Satellite Image', color=TEXT_COL, fontsize=13)
plt.tight_layout(); plt.show()


## 4. Edge Detection & Field Boundary Extraction

Three complementary edge signals are combined:
1. **Canny** on blurred grayscale (two blur radii)
2. **Multi-channel Sobel** (computed separately on R, G, B)
3. **Morphological closing** to bridge gaps along field edges


In [ ]:
def compute_edge_map(img_rgb, gray, cfg):
    """Multi-channel edge map combining Canny + Sobel."""
    blur3 = cv2.GaussianBlur(gray, (3,3), 0)
    blur5 = cv2.GaussianBlur(gray, (5,5), 0)
    
    canny1 = cv2.Canny(blur3, cfg['edge_canny_low'], cfg['edge_canny_high'])
    canny2 = cv2.Canny(blur5, int(cfg['edge_canny_low']*0.7),
                               int(cfg['edge_canny_high']*0.7))
    
    # Sobel across all 3 channels
    edges_ch = np.zeros(gray.shape, dtype=np.float32)
    for c in range(3):
        ch = img_rgb[:,:,c].astype(np.float32)
        gx = cv2.Sobel(ch, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(ch, cv2.CV_32F, 0, 1, ksize=3)
        edges_ch += np.sqrt(gx**2 + gy**2)
    edges_ch = cv2.normalize(edges_ch, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, strong = cv2.threshold(edges_ch, 28, 255, cv2.THRESH_BINARY)
    
    combined = cv2.bitwise_or(cv2.bitwise_or(canny1, canny2), strong)
    
    # Close horizontal and vertical gaps
    kh = cv2.getStructuringElement(cv2.MORPH_RECT, (7,1))
    kv = cv2.getStructuringElement(cv2.MORPH_RECT, (1,7))
    k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    edge_clean = cv2.bitwise_or(
        cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kh),
        cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kv)
    )
    edge_clean = cv2.dilate(edge_clean, k3, iterations=1)
    return edge_clean, edges_ch

edge_map, sobel_mag = compute_edge_map(img_rgb, gray, CONFIG)
print(f"Edge pixels detected: {int((edge_map > 0).sum()):,}  "
      f"({(edge_map>0).sum()/(H*W)*100:.1f}% of image)")

# Hough line detection on the clean edge map
lines = cv2.HoughLinesP(edge_map, 1, np.pi/180,
                         threshold=CONFIG['hough_threshold'],
                         minLineLength=CONFIG['hough_min_length'],
                         maxLineGap=CONFIG['hough_max_gap'])
print(f"Hough lines detected: {len(lines) if lines is not None else 0}")

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(17, 5), facecolor=BG_DARK)

axes[0].imshow(sobel_mag, cmap='inferno'); axes[0].axis('off')
axes[0].set_title('Multi-channel Sobel Magnitude', color=TEXT_COL, fontsize=10)

edge_col = img_rgb.copy()
edge_col[edge_map > 0] = [255, 230, 40]
axes[1].imshow(edge_col); axes[1].axis('off')
axes[1].set_title('Combined Edge Map (yellow)', color=TEXT_COL, fontsize=10)

hough_vis = img_rgb.copy()
if lines is not None:
    for l in lines:
        x1,y1,x2,y2 = l[0]
        lw = 2 if np.hypot(x2-x1,y2-y1)>70 else 1
        cv2.line(hough_vis,(x1,y1),(x2,y2),(255,60,60),lw,cv2.LINE_AA)
axes[2].imshow(hough_vis); axes[2].axis('off')
axes[2].set_title(f'Hough Lines ({len(lines) if lines else 0} detected)',
                  color=TEXT_COL, fontsize=10)

plt.suptitle('Edge Detection Pipeline', color=TEXT_COL, fontsize=13)
plt.tight_layout(); plt.show()


## 5. Felzenszwalb Graph-Based Segmentation

Felzenszwalb's algorithm builds a minimum spanning tree of the image graph and  
merges regions based on internal vs. inter-region contrast — well suited for  
agricultural parcels which have coherent internal texture.

Three scales capture: sub-parcel detail → parcel level → coarse land blocks.


In [ ]:
def colorize_seg(seg_map, img_rgb, palette=PALETTE, alpha=0.52,
                 border_col=(255,255,255), border_w=2, label_parcels=True,
                 min_pct=1.5):
    """Blend segment colours onto image and draw crisp boundaries."""
    H, W = img_rgb.shape[:2]
    total = H * W
    out = img_rgb.copy().astype(np.float32)
    
    # Sort segments by area (largest first → most distinct colour)
    areas = sorted([(s, (seg_map==s).sum()) for s in range(seg_map.max()+1)],
                   key=lambda x: -x[1])
    col_map = {s: palette[i % len(palette)] for i, (s, _) in enumerate(areas)}
    
    for s, col in col_map.items():
        mask = seg_map == s
        for c in range(3):
            out[:,:,c][mask] = out[:,:,c][mask]*(1-alpha) + col[c]*alpha
    out = np.clip(out, 0, 255).astype(np.uint8)
    
    # Boundary mask
    bnd = np.zeros((H,W), dtype=bool)
    for dy,dx in [(-1,0),(1,0),(0,-1),(0,1)]:
        bnd |= (seg_map != np.roll(np.roll(seg_map,dy,0),dx,1))
    bnd8 = cv2.dilate(bnd.astype(np.uint8)*255, np.ones((border_w,border_w),np.uint8))
    out[bnd8 > 0] = border_col
    
    return out, col_map, areas

# Run at three scales
fz_fine   = felzenszwalb(img_f, scale=CONFIG['fz_scale_fine'],
                          sigma=CONFIG['fz_sigma'], min_size=CONFIG['fz_min_size_fine'])
fz_mid    = felzenszwalb(img_f, scale=CONFIG['fz_scale_mid'],
                          sigma=CONFIG['fz_sigma'], min_size=CONFIG['fz_min_size_mid'])
fz_coarse = felzenszwalb(img_f, scale=CONFIG['fz_scale_coarse'],
                          sigma=CONFIG['fz_sigma']+0.3, min_size=CONFIG['fz_min_size_crs'])

print(f"Segments  — fine: {fz_fine.max()+1} | mid: {fz_mid.max()+1} | coarse: {fz_coarse.max()+1}")

fz_fine_v,   cm_fine,   ar_fine   = colorize_seg(fz_fine,   img_rgb, alpha=0.48)
fz_mid_v,    cm_mid,    ar_mid    = colorize_seg(fz_mid,    img_rgb, alpha=0.52)
fz_coarse_v, cm_coarse, ar_coarse = colorize_seg(fz_coarse, img_rgb, alpha=0.55)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG_DARK)
titles = [
    f'Fine  (scale={CONFIG["fz_scale_fine"]})  →  {fz_fine.max()+1} segments',
    f'Mid   (scale={CONFIG["fz_scale_mid"]})  →  {fz_mid.max()+1} segments',
    f'Coarse (scale={CONFIG["fz_scale_coarse"]}) →  {fz_coarse.max()+1} segments',
]
for ax, img, title in zip(axes, [fz_fine_v, fz_mid_v, fz_coarse_v], titles):
    ax.imshow(img); ax.axis('off')
    ax.set_title(title, color=TEXT_COL, fontsize=10)

# Label parcels on mid panel
for idx, (s, area) in enumerate(ar_mid[:14]):
    mask = fz_mid == s
    ys, xs = np.where(mask)
    if len(ys) < 20: continue
    cy, cx = int(ys.mean()), int(xs.mean())
    col = cm_mid[s]
    axes[1].text(cx, cy, f'P{idx+1}\n{area/(H*W)*100:.1f}%',
                 ha='center', va='center', fontsize=7.5, fontweight='500',
                 color=f'#{col[0]:02x}{col[1]:02x}{col[2]:02x}',
                 bbox=dict(fc='#000000aa', ec='none', pad=1, boxstyle='round,pad=0.2'))

plt.suptitle('Felzenszwalb Multi-Scale Segmentation', color=TEXT_COL, fontsize=13)
plt.tight_layout(); plt.show()


## 6. SLIC Superpixel Segmentation

Simple Linear Iterative Clustering (SLIC) partitions the image into compact,  
perceptually uniform superpixels. High `compactness` enforces square-ish  
superpixels that respect field grid structure.


In [ ]:
slic_seg = slic(img_f,
                n_segments = CONFIG['slic_n_segments'],
                compactness= CONFIG['slic_compactness'],
                sigma      = CONFIG['slic_sigma'],
                start_label=1, channel_axis=2)

print(f"SLIC superpixels generated: {slic_seg.max()}")

# Mark boundaries on original
slic_vis = mark_boundaries(img_f, slic_seg, color=(0, 0.9, 1.0), mode='thick')
slic_col, _, _ = colorize_seg(slic_seg, img_rgb, alpha=0.45,
                               border_col=(200,220,255), border_w=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG_DARK)
axes[0].imshow(slic_vis); axes[0].axis('off')
axes[0].set_title(f'SLIC boundaries (cyan)  —  {slic_seg.max()} superpixels',
                  color=TEXT_COL, fontsize=11)
axes[1].imshow(slic_col); axes[1].axis('off')
axes[1].set_title('SLIC — colour-filled regions', color=TEXT_COL, fontsize=11)
plt.suptitle('SLIC Superpixel Segmentation', color=TEXT_COL, fontsize=13)
plt.tight_layout(); plt.show()


## 7. Watershed Segmentation

Distance transform on the inverted edge map identifies parcel interiors (sure foreground).  
Connected component markers seed the watershed flood-fill.  
Watershed boundary pixels are marked **red** (label = −1).


In [ ]:
def run_watershed(edge_map, img_bgr, fg_thresh=0.35):
    """Watershed segmentation driven by edge-map distance transform."""
    edge_inv = cv2.bitwise_not(edge_map)
    dist = cv2.distanceTransform(edge_inv, cv2.DIST_L2, 5)
    cv2.normalize(dist, dist, 0, 1.0, cv2.NORM_MINMAX)
    
    _, sure_fg = cv2.threshold(dist, fg_thresh, 1.0, cv2.THRESH_BINARY)
    sure_fg = sure_fg.astype(np.uint8) * 255
    
    n_labels, markers = cv2.connectedComponents(sure_fg)
    markers += 1
    markers[edge_map > 0] = 0
    
    ws = img_bgr.copy()
    cv2.watershed(ws, markers)
    return markers, n_labels, dist

markers_ws, n_ws, dist_map = run_watershed(edge_map, img_bgr,
                                            fg_thresh=CONFIG['ws_fg_threshold'])
print(f"Watershed regions: {markers_ws.max()}  (including background)")

# Visualise
ws_vis = img_rgb.copy().astype(np.float32)
for s in range(1, markers_ws.max()+1):
    mask = markers_ws == s
    if mask.sum() == 0: continue
    col = PALETTE[(s-1) % len(PALETTE)]
    for c in range(3):
        ws_vis[:,:,c][mask] = ws_vis[:,:,c][mask]*0.5 + col[c]*0.5
ws_vis = np.clip(ws_vis, 0, 255).astype(np.uint8)
ws_vis[markers_ws == -1] = [255, 60, 60]   # red watershed boundaries

fig, axes = plt.subplots(1, 3, figsize=(17, 5), facecolor=BG_DARK)
axes[0].imshow(dist_map, cmap='viridis'); axes[0].axis('off')
axes[0].set_title('Distance Transform', color=TEXT_COL, fontsize=10)
axes[1].imshow(ws_vis); axes[1].axis('off')
axes[1].set_title(f'Watershed — {markers_ws.max()} regions (red = boundaries)',
                  color=TEXT_COL, fontsize=10)

# Overlay boundaries on original
ws_overlay = img_rgb.copy()
ws_overlay[markers_ws == -1] = [255, 60, 60]
axes[2].imshow(ws_overlay); axes[2].axis('off')
axes[2].set_title('Watershed boundaries on original', color=TEXT_COL, fontsize=10)
plt.suptitle('Watershed Segmentation', color=TEXT_COL, fontsize=13)
plt.tight_layout(); plt.show()


## 8. K-Means Spectral Classification

K-means clusters pixels by their RGB colour (spectral signature) into K classes.  
Each class corresponds to a distinct land-cover type: bare soil, crop residue,  
tilled soil, green vegetation, moist soil, etc.


In [ ]:
K = CONFIG['kmeans_k']

# K-means on flattened RGB pixels
pixels   = img_rgb.reshape(-1, 3).astype(np.float32)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 0.5)
_, labels_km, centers_km = cv2.kmeans(pixels, K, None, criteria,
                                       8, cv2.KMEANS_RANDOM_CENTERS)
km_seg    = labels_km.reshape(H, W)
centers_km = centers_km.astype(np.uint8)

# Assign semantic names based on mean brightness & greenness
class_info = {}
for k in range(K):
    mask = km_seg == k
    mean_col = img_rgb[mask].mean(axis=0) if mask.sum() > 0 else np.zeros(3)
    brightness = mean_col.mean()
    greenness  = float(mean_col[1]) - float(mean_col[0])
    class_info[k] = {'mean': mean_col, 'brightness': brightness,
                     'greenness': greenness, 'area_pct': mask.sum()/(H*W)*100}

# Sort by brightness for label assignment
sorted_k = sorted(class_info.items(), key=lambda x: x[1]['brightness'])
LAND_LABELS = ['Dark moist soil','Dry crop residue','Mid-tone tilled',
               'Pale tilled soil','Light bare soil','Very pale/sand']

km_color_map = {}
for rank, (k, _) in enumerate(sorted_k):
    km_color_map[k] = (rank, LAND_LABELS[rank % len(LAND_LABELS)])

# Build coloured output
CLASS_COLORS = np.array([
    [  40, 100,  60],  # dark moist
    [ 120, 100,  70],  # dry residue
    [ 160, 140, 110],  # mid tilled
    [ 210, 185, 150],  # pale tilled
    [ 235, 215, 185],  # light bare
    [ 255, 245, 225],  # very pale
], dtype=np.uint8)

km_rank_map = np.zeros((H,W), dtype=np.uint8)
for k, (rank, _) in km_color_map.items():
    km_rank_map[km_seg == k] = rank
km_rgb_out = CLASS_COLORS[km_rank_map]

print("K-Means class summary:")
for k, (rank, label) in sorted(km_color_map.items(), key=lambda x: x[1][0]):
    info = class_info[k]
    print(f"  Class {k} [{label:<22}]: {info['area_pct']:5.1f}% "
          f"  brightness={info['brightness']:.0f}  greenness={info['greenness']:+.1f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG_DARK)
axes[0].imshow(km_rgb_out); axes[0].axis('off')
axes[0].set_title(f'K-Means Spectral Classes (k={K})', color=TEXT_COL, fontsize=11)

# Legend
patches = [mpatches.Patch(color=CLASS_COLORS[i]/255, label=LAND_LABELS[i])
           for i in range(K)]
axes[0].legend(handles=patches, loc='lower left', fontsize=8, frameon=True,
               facecolor='#1a1a28', edgecolor='#334', labelcolor='white')

# Blend with original
km_blend = cv2.addWeighted(img_rgb, 0.45, km_rgb_out, 0.55, 0)
axes[1].imshow(km_blend); axes[1].axis('off')
axes[1].set_title('K-Means blend with original', color=TEXT_COL, fontsize=11)
plt.suptitle('K-Means Spectral Classification', color=TEXT_COL, fontsize=13)
plt.tight_layout(); plt.show()


## 9. Full Composite — Best-of-All-Methods Output

Combines:
- **Coarse Felzenszwalb** fill colours (land block identity)
- **Fine Felzenszwalb** boundary lines (white, 2px)
- **Sub-segment** boundaries from mid-scale (light gray, 1px)
- **Yellow** parcel boundary lines from the edge map
- **Parcel ID labels** (P1–P12) with area percentages


In [ ]:
# Build composite
comp = img_rgb.copy().astype(np.float32)

# Tint by coarse FZ regions
n_coarse = fz_coarse.max() + 1
for s in range(n_coarse):
    mask = fz_coarse == s
    if mask.sum() == 0: continue
    col = PALETTE[s % len(PALETTE)]
    for c in range(3):
        comp[:,:,c][mask] = comp[:,:,c][mask]*0.62 + col[c]*0.38

comp_u8 = np.clip(comp, 0, 255).astype(np.uint8)

# Fine boundaries white
bnd_fine = np.zeros((H,W), dtype=bool)
for dy,dx in [(-1,0),(1,0),(0,-1),(0,1)]:
    bnd_fine |= (fz_mid != np.roll(np.roll(fz_mid,dy,0),dx,1))
bnd_fine8 = cv2.dilate(bnd_fine.astype(np.uint8)*255, np.ones((2,2),np.uint8))
comp_u8[bnd_fine8 > 0] = [255,255,255]

# Sub-boundaries light gray
bnd_sub = np.zeros((H,W), dtype=bool)
for dy,dx in [(-1,0),(1,0),(0,-1),(0,1)]:
    bnd_sub |= (fz_fine != np.roll(np.roll(fz_fine,dy,0),dx,1))
comp_u8[bnd_sub & ~bnd_fine] = [160,165,190]

# Edge map lines yellow
comp_u8[edge_map > 0] = [255,240,50]

# Labels
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=BG_DARK)
axes[0].imshow(img_rgb); axes[0].axis('off')
axes[0].set_title('Original', color=TEXT_COL, fontsize=12)

axes[1].imshow(comp_u8); axes[1].axis('off')
axes[1].set_title(f'Composite — {fz_mid.max()+1} parcels identified',
                  color=TEXT_COL, fontsize=12)

# Parcel labels
for idx, (s, area) in enumerate(ar_mid[:14]):
    mask = fz_mid == s
    ys, xs = np.where(mask)
    if len(ys) < 20: continue
    cy, cx = int(ys.mean()), int(xs.mean())
    col = cm_mid[s]
    axes[1].text(cx, cy, f'P{idx+1}\n{area/(H*W)*100:.1f}%',
                 ha='center', va='center', fontsize=8, fontweight='500',
                 color=f'#{col[0]:02x}{col[1]:02x}{col[2]:02x}',
                 bbox=dict(fc='#000000bb', ec='none', pad=1.5, boxstyle='round,pad=0.3'))

legend_items = [
    mpatches.Patch(fc='white',          ec='gray', lw=1, label='Primary parcel boundaries (FZ-mid)'),
    mpatches.Patch(fc='#a0a5be',        ec='gray', lw=1, label='Sub-parcel boundaries (FZ-fine)'),
    mpatches.Patch(fc='#fff032',        ec='gray', lw=1, label='Edge-detected field lines'),
]
axes[1].legend(handles=legend_items, loc='lower left', fontsize=8, frameon=True,
               facecolor='#1a1a28', edgecolor='#334', labelcolor='white', handlelength=1.2)

plt.suptitle('Final Composite Segmentation', color=TEXT_COL, fontsize=14, fontweight='500')
plt.tight_layout(); plt.show()


## 10. Parcel Statistics & Summary Report

In [ ]:
print("=" * 62)
print("  AGRICULTURAL PARCEL SEGMENTATION — SUMMARY REPORT")
print("=" * 62)
print(f"  Image dimensions : {W} × {H} pixels")
print(f"  Total area       : {W*H:,} pixels")
print()
print("  ── SEGMENTATION METHOD RESULTS ──")
print(f"  Felzenszwalb fine   (scale={CONFIG['fz_scale_fine']:>3}) : {fz_fine.max()+1:>4} segments")
print(f"  Felzenszwalb mid    (scale={CONFIG['fz_scale_mid']:>3}) : {fz_mid.max()+1:>4} segments  ← primary")
print(f"  Felzenszwalb coarse (scale={CONFIG['fz_scale_coarse']:>3}) : {fz_coarse.max()+1:>4} segments")
print(f"  SLIC superpixels                    : {slic_seg.max():>4}")
print(f"  Watershed regions                   : {markers_ws.max():>4}")
print(f"  K-Means classes                     : {K:>4}")
print(f"  Hough field lines                   : {len(lines) if lines is not None else 0:>4}")
print()
print("  ── TOP PARCELS (Felzenszwalb mid-scale) ──")
print(f"  {'#':<4} {'Area (px)':>10} {'Area (%)':>9}  Mean RGB")
print(f"  {'-'*46}")
for idx, (s, area) in enumerate(ar_mid[:12]):
    mask = fz_mid == s
    if mask.sum() == 0: continue
    mean_col = img_rgb[mask].mean(axis=0)
    print(f"  P{idx+1:<3} {area:>10,} {area/(H*W)*100:>8.1f}%  "
          f"RGB({mean_col[0]:.0f}, {mean_col[1]:.0f}, {mean_col[2]:.0f})")
print()
print("  ── K-MEANS LAND COVER ──")
for k, (rank, label) in sorted(km_color_map.items(), key=lambda x: x[1][0]):
    info = class_info[k]
    bar = '█' * int(info['area_pct'] / 2)
    print(f"  {label:<24}: {info['area_pct']:5.1f}%  {bar}")
print("=" * 62)


## 11. Export Results

Save all segmentation outputs as GeoTIFF-compatible arrays and PNG images.


In [ ]:
import os
out_dir = "segmentation_outputs"
os.makedirs(out_dir, exist_ok=True)

# Save PNG outputs
outputs = {
    "01_original"           : img_rgb,
    "02_edge_map"           : edge_col,
    "03_fz_fine"            : fz_fine_v,
    "04_fz_mid_parcels"     : fz_mid_v,
    "05_fz_coarse"          : fz_coarse_v,
    "06_slic_coloured"      : slic_col,
    "07_watershed"          : ws_vis,
    "08_kmeans_spectral"    : km_blend,
    "09_composite_final"    : comp_u8,
}

for name, arr in outputs.items():
    path = os.path.join(out_dir, f"{name}.png")
    Image.fromarray(arr.astype(np.uint8)).save(path)
    print(f"  Saved: {path}")

# Save segmentation label arrays as NPY
np.save(os.path.join(out_dir, "fz_mid_labels.npy"),    fz_mid)
np.save(os.path.join(out_dir, "fz_coarse_labels.npy"), fz_coarse)
np.save(os.path.join(out_dir, "slic_labels.npy"),      slic_seg)
np.save(os.path.join(out_dir, "kmeans_labels.npy"),    km_seg)
np.save(os.path.join(out_dir, "watershed_labels.npy"), markers_ws)

print(f"\n✅ All outputs saved to: {out_dir}/")
print(f"   PNG images  : {len(outputs)}")
print(f"   Label arrays: 5 (.npy)")


## References & Further Reading

| Method | Citation |
|--------|----------|
| Felzenszwalb | Felzenszwalb, P. & Huttenlocher, D. (2004). Efficient graph-based image segmentation. *IJCV* 59(2):167–181 |
| SLIC | Achanta, R. et al. (2012). SLIC superpixels compared to state-of-the-art superpixel methods. *IEEE TPAMI* 34(11):2274–2282 |
| Watershed | Vincent, L. & Soille, P. (1991). Watersheds in digital spaces. *IEEE TPAMI* 13(6):583–598 |
| K-Means | Lloyd, S.P. (1982). Least squares quantization in PCM. *IEEE Trans. Inf. Theory* 28(2):129–137 |
| Hough | Duda, R. & Hart, P. (1972). Use of the Hough transformation to detect lines and curves. *CACM* 15(1):11–15 |

**Related work by the author:**  
Ahmed, Z.U. et al. (2022). Machine learning approaches for crop yield prediction and agricultural remote sensing analysis. *Field Crops Research* (see [zia207/GitHub](https://github.com/zia207))

---
*Notebook created with Uppata Analytics LLC — [uppata.com](https://uppata.com)*
